In [5]:
import json, os
import networkx as nx
from datetime import datetime, timezone
from collections import defaultdict

# Load alerts
alerts = []
with open(r"D:\Xbrain\Phase 2 AIops\w2\d1\dataset\alerts_sample.jsonl") as f:
    for line in f:
        line = line.strip()
        if line:
            alerts.append(json.loads(line))

# Load service graph — dùng "from"/"to" chứ không phải "source"/"target"
with open(r"D:\Xbrain\Phase 2 AIops\w2\d1\dataset\services.json") as f:
    svc_data = json.load(f)

graph = nx.DiGraph()
for edge in svc_data["edges"]:
    graph.add_edge(edge["from"], edge["to"])   # ← đúng key

print(f"Loaded {len(alerts)} alerts")
print(f"Graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")
print(f"\nSample alert:\n{json.dumps(alerts[0], indent=2)}")

Loaded 20 alerts
Graph: 14 nodes, 17 edges

Sample alert:
{
  "id": "a-0001",
  "ts": "2026-06-12T09:42:01Z",
  "service": "payment-svc",
  "metric": "db_connection_pool_used_ratio",
  "severity": "warn",
  "value": 0.85,
  "threshold": 0.8,
  "labels": {
    "env": "prod",
    "region": "ap-southeast-1"
  }
}


In [6]:
def fingerprint(alert):
    return f"{alert['service']}|{alert['metric']}|{alert['severity']}"

class Deduper:
    def __init__(self):
        self.store = {}
    
    def push(self, alert):
        fp = fingerprint(alert)
        if fp not in self.store:
            self.store[fp] = {
                'count': 1,
                'first_seen': alert['ts'],
                'last_seen': alert['ts'],
                'alert_ids': [alert['id']]
            }
        else:
            c = self.store[fp]
            c['count'] += 1
            c['last_seen'] = alert['ts']
            c['alert_ids'].append(alert['id'])
        return fp

# Test dedup
deduper = Deduper()
for a in alerts:
    deduper.push(a)

print(f"Unique fingerprints: {len(deduper.store)}")
for fp, info in deduper.store.items():
    if info['count'] > 1:
        print(f"  DUPLICATE x{info['count']}: {fp}")

Unique fingerprints: 17
  DUPLICATE x2: payment-svc|db_connection_pool_used_ratio|crit
  DUPLICATE x3: payment-svc|latency_p99_ms|crit


In [7]:
def parse_ts(ts_str):
    return datetime.fromisoformat(ts_str.replace('Z', '+00:00'))

def session_groups(alerts, gap_sec=120):
    if not alerts:
        return []
    sorted_alerts = sorted(alerts, key=lambda a: a['ts'])
    groups = [[sorted_alerts[0]]]
    
    for alert in sorted_alerts[1:]:
        last_ts = parse_ts(groups[-1][-1]['ts'])
        curr_ts = parse_ts(alert['ts'])
        gap = (curr_ts - last_ts).total_seconds()
        
        if gap <= gap_sec:
            groups[-1].append(alert)
        else:
            groups.append([alert])
    
    return groups

# Test với gap_sec=120 (sweet spot theo bài)
sessions = session_groups(alerts, gap_sec=120)
print(f"Sessions formed: {len(sessions)}")
for i, s in enumerate(sessions):
    print(f"  Session {i}: {len(s)} alerts | "
          f"{s[0]['ts']} → {s[-1]['ts']}")

Sessions formed: 1
  Session 0: 20 alerts | 2026-06-12T09:42:01Z → 2026-06-12T09:48:30Z


In [11]:
def topology_group(alerts, graph, max_hop=2):
    undirected = graph.to_undirected()
    by_service = defaultdict(list)
    for a in alerts:
        by_service[a['service']].append(a)
    
    services = list(by_service.keys())
    parent = {s: s for s in services}
    
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    
    def union(x, y):
        parent[find(x)] = find(y)
    
    for i, s1 in enumerate(services):
        for s2 in services[i+1:]:
            # Service không có trong graph → skip
            if not graph.has_node(s1) or not graph.has_node(s2):
                continue
            try:
                dist = nx.shortest_path_length(undirected, s1, s2)
                if dist <= max_hop:
                    union(s1, s2)
            except nx.NetworkXNoPath:
                pass
    
    groups = defaultdict(list)
    for s in services:
        groups[find(s)].extend(by_service[s])
    
    return list(groups.values())

# Test
test_session = sessions[0] if sessions else alerts
topo_groups = topology_group(test_session, graph, max_hop=2)
print(f"Alerts in session 0: {len(test_session)}")
print(f"Topology groups: {len(topo_groups)}")
for i, g in enumerate(topo_groups):
    svcs = sorted({a['service'] for a in g})
    print(f"  Group {i}: {svcs}")

Alerts in session 0: 20
Topology groups: 1
  Group 0: ['cart-svc', 'checkout-svc', 'edge-lb', 'notification-svc', 'payment-svc', 'recommender-svc', 'search-svc']


In [17]:
SEVERITY_RANK = {"info": 0, "warn": 1, "crit": 2}

def max_severity(group):
    return max(
        (a['severity'] for a in group),
        key=lambda s: SEVERITY_RANK.get(s, 0)
    )

def correlate(alerts, graph, gap_sec=120, max_hop=2):
    sessions = session_groups(alerts, gap_sec=gap_sec)
    clusters = []
    
    for s_idx, session_alerts in enumerate(sessions):
        topo_groups = topology_group(session_alerts, graph, max_hop=max_hop)
        for g_idx, group in enumerate(topo_groups):
            clusters.append({
                'cluster_id': f'c-{s_idx:03d}-{g_idx:03d}',
                'alert_count': len(group),
                'services': sorted({a['service'] for a in group}),
                'time_range': [
                    min(a['ts'] for a in group),
                    max(a['ts'] for a in group)
                ],
                'max_severity': max_severity(group),
                'fingerprints': sorted({fingerprint(a) for a in group})
            })
    
    return clusters

# Chạy pipeline
clusters = correlate(alerts, graph, gap_sec=30, max_hop=2)
reduction = 1 - len(clusters) / len(alerts)

print(f"Input alerts   : {len(alerts)}")
print(f"Output clusters: {len(clusters)}")
print(f"Reduction ratio: {reduction:.2f}  (cần >= 0.5)")
print()
for c in clusters:
    print(f"  {c['cluster_id']} | {c['alert_count']} alerts | "
          f"severity={c['max_severity']} | services={c['services']}")

# Ghi file
import os
os.makedirs(r"D:\Xbrain\Phase 2 AIops\w2\d1\results", exist_ok=True)

output = {
    "input_alerts": len(alerts),
    "output_clusters": len(clusters),
    "reduction_ratio": round(reduction, 2),
    "clusters": clusters
}

out_path = r"D:\Xbrain\Phase 2 AIops\w2\d1\results\cluster_summary.json"
with open(out_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"\nSaved → {out_path}")

Input alerts   : 20
Output clusters: 5
Reduction ratio: 0.75  (cần >= 0.5)

  c-000-000 | 12 alerts | severity=crit | services=['cart-svc', 'checkout-svc', 'edge-lb', 'notification-svc', 'payment-svc']
  c-001-000 | 1 alerts | severity=warn | services=['recommender-svc']
  c-002-000 | 2 alerts | severity=crit | services=['edge-lb', 'payment-svc']
  c-003-000 | 2 alerts | severity=crit | services=['checkout-svc', 'search-svc']
  c-004-000 | 3 alerts | severity=crit | services=['edge-lb', 'notification-svc', 'payment-svc']

Saved → D:\Xbrain\Phase 2 AIops\w2\d1\results\cluster_summary.json


In [16]:
print("=== SESSIONS với gap_sec=60 ===")
sessions_60 = session_groups(alerts, gap_sec=60)
for i, s in enumerate(sessions_60):
    print(f"Session {i}: {len(s)} alerts | {s[0]['ts']} → {s[-1]['ts']}")
    print(f"  services: {sorted({a['service'] for a in s})}")

print("\n=== SESSIONS với gap_sec=30 ===")
sessions_30 = session_groups(alerts, gap_sec=30)
for i, s in enumerate(sessions_30):
    print(f"Session {i}: {len(s)} alerts | {s[0]['ts']} → {s[-1]['ts']}")
    print(f"  services: {sorted({a['service'] for a in s})}")

print("\n=== Khoảng cách giữa các alert liên tiếp ===")
sorted_alerts = sorted(alerts, key=lambda a: a['ts'])
for i in range(1, len(sorted_alerts)):
    t1 = parse_ts(sorted_alerts[i-1]['ts'])
    t2 = parse_ts(sorted_alerts[i]['ts'])
    gap = (t2 - t1).total_seconds()
    print(f"  {sorted_alerts[i-1]['id']} → {sorted_alerts[i]['id']}: {gap:.0f}s")

=== SESSIONS với gap_sec=60 ===
Session 0: 20 alerts | 2026-06-12T09:42:01Z → 2026-06-12T09:48:30Z
  services: ['cart-svc', 'checkout-svc', 'edge-lb', 'notification-svc', 'payment-svc', 'recommender-svc', 'search-svc']

=== SESSIONS với gap_sec=30 ===
Session 0: 12 alerts | 2026-06-12T09:42:01Z → 2026-06-12T09:44:30Z
  services: ['cart-svc', 'checkout-svc', 'edge-lb', 'notification-svc', 'payment-svc']
Session 1: 1 alerts | 2026-06-12T09:45:10Z → 2026-06-12T09:45:10Z
  services: ['recommender-svc']
Session 2: 2 alerts | 2026-06-12T09:45:42Z → 2026-06-12T09:46:01Z
  services: ['edge-lb', 'payment-svc']
Session 3: 2 alerts | 2026-06-12T09:46:50Z → 2026-06-12T09:47:12Z
  services: ['checkout-svc', 'search-svc']
Session 4: 3 alerts | 2026-06-12T09:48:01Z → 2026-06-12T09:48:30Z
  services: ['edge-lb', 'notification-svc', 'payment-svc']

=== Khoảng cách giữa các alert liên tiếp ===
  a-0001 → a-0002: 17s
  a-0002 → a-0003: 4s
  a-0003 → a-0004: 8s
  a-0004 → a-0005: 15s
  a-0005 → a-0006: 16